# Challenge 2: Quantum Coin Duel ★☆☆

**The game.** A coin sits hidden in a box, heads up. Two players take turns: the Quantum player, then the Classical player, then the Quantum player again. On each turn a player can flip the coin or leave it alone, without looking. When the box opens, the Quantum player wins if the coin is heads.

With normal coins, nobody can do better than 50/50. You'll show that a Quantum player who knows one trick wins **every single time**, then figure out why, then find the counter-move.

**Quantum ideas:** superposition, interference, the Bloch sphere.
**Best for:** game lovers and puzzle people.
**Time plan:** Missions 1 to 5 take about 60 to 75 minutes. Spend the rest on "Make it yours."

This game comes from a famous 1999 paper by physicist David Meyer.

**In the circuit:** heads = `0`, tails = `1`, "flip" = an X gate, and "leave it" = do nothing.

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
def check_mission_1(classical_move):
    try:
        a = QuantumCircuit(1); classical_move(a, True)
        b = QuantumCircuit(1); classical_move(b, False)
        ok = _close(probabilities(a), {"1": 1.0}) and _close(probabilities(b), {"0": 1.0})
    except Exception:
        ok = False
    _report(ok, "Mission 1 complete: the classical move works.",
            "inside classical_move, write: if flip: then qc.x(0) on the next line (indented).")

def check_mission_2(play_quantum_game):
    try:
        games = [play_quantum_game(f) for f in [True, False]]
        ok = all(g.count_ops().get("h", 0) == 2 and _close(probabilities(g), {"0": 1.0}) for g in games)
    except Exception:
        ok = False
    _report(ok, "Mission 2 complete: the quantum player wins 100% of the time!",
            "the quantum move is a single H gate on qubit 0.")

def check_mission_4(counter_move):
    try:
        qc = QuantumCircuit(1); qc.h(0); counter_move(qc); qc.h(0)
        ok = _close(probabilities(qc), {"1": 1.0})
    except Exception:
        ok = False
    _report(ok, "Mission 4 complete: you beat the quantum strategy!",
            "X does nothing to a spinning coin. Try a gate that changes the hidden phase.")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## Mission 1: The classical game

First, play the normal version where everyone just flips or doesn't, at random.

**TODO:** Finish `classical_move` so it flips the coin (X gate) when `flip` is `True`.

In [ ]:
def classical_move(qc, flip):
    """Flip the coin with an X gate if flip is True. Otherwise leave it alone."""
    # TODO: if flip is True, apply an X gate to qubit 0
    pass

def play_classical_game():
    qc = QuantumCircuit(1)                               # the coin starts heads (0)
    classical_move(qc, random.choice([True, False]))     # Player A
    classical_move(qc, random.choice([True, False]))     # Player B
    classical_move(qc, random.choice([True, False]))     # Player A again
    qc.measure_all()
    return run_once(qc)        # '0' = heads (A wins), '1' = tails (B wins)

results = [play_classical_game() for i in range(200)]
print("Player A won", results.count("0"), "out of 200 games")

In [ ]:
check_mission_1(classical_move)

About 100 out of 200: a coin flip. Nobody has an edge.

## Mission 2: The quantum player

Now give Player A a quantum power: on both of their turns, they apply an **H gate** instead of flipping. The Classical player still chooses flip or stay.

**TODO:** Make `quantum_move` apply an H gate to qubit 0.

In [ ]:
def quantum_move(qc):
    # TODO: the Quantum player's secret move is an H gate on qubit 0
    pass

def play_quantum_game(classical_flip):
    qc = QuantumCircuit(1)
    quantum_move(qc)                     # Quantum player
    classical_move(qc, classical_flip)   # Classical player: flip or stay
    quantum_move(qc)                     # Quantum player again
    qc.measure_all()
    return qc

wins = 0
for game in range(200):
    qc = play_quantum_game(random.choice([True, False]))
    if run_once(qc) == "0":
        wins += 1
print("Quantum player won", wins, "out of 200 games")

In [ ]:
check_mission_2(play_quantum_game)

## Mission 3: Why does that work?

Watch the qubit's arrow on the Bloch sphere after every turn. Up is heads, down is tails. Run this cell and compare the two games.

In [ ]:
for classical_flip in [False, True]:
    print(f"===== Classical player chooses flip = {classical_flip} =====")
    qc = QuantumCircuit(1)
    quantum_move(qc)
    display(plot_bloch_multivector(Statevector(qc), title="After the Quantum player's 1st move"))
    classical_move(qc, classical_flip)
    display(plot_bloch_multivector(Statevector(qc), title=f"After the Classical player (flip={classical_flip})"))
    quantum_move(qc)
    display(plot_bloch_multivector(Statevector(qc), title="After the Quantum player's 2nd move"))

**What happened:** After the first H, the arrow points sideways along the X axis. The X gate rotates the arrow around that same axis, so an arrow already pointing along it **doesn't move**. Flipping a perfectly spinning coin changes nothing. The second H turns the arrow back up to heads.

**Discuss with your team:** How would you explain this to a friend who has never heard of a qubit? Write it down; it's part of your demo.

## Mission 4: Fight back

Now the Classical player gets quantum powers too. They may use exactly one gate on their turn: `qc.x(0)`, `qc.y(0)`, `qc.z(0)`, or `qc.h(0)`.

**TODO:** Find a gate that makes the Quantum player lose every time. Experiment: change the gate, run, and look at the odds.

In [ ]:
def counter_move(qc):
    # TODO: try different gates until the Quantum player always loses
    qc.x(0)

qc = QuantumCircuit(1)
quantum_move(qc)
counter_move(qc)
quantum_move(qc)
qc.measure_all()
print("Odds:", probabilities(qc), "   ('1' means tails, so the Quantum player loses)")

In [ ]:
check_mission_4(counter_move)

Use the Bloch sphere code from Mission 3 to see why your counter-move works. A quantum strategy only beats players who are stuck with classical moves.

## Mission 5: Play against the quantum computer

Now you play the Classical player. Type `flip` or `stay` when asked.

**TODO:** Apply your move with `classical_move`. `flip` should be `True` if you typed `flip`.

In [ ]:
def play_vs_quantum():
    qc = QuantumCircuit(1)
    quantum_move(qc)
    choice = input("The coin is in the box. Type 'flip' or 'stay': ").strip().lower()
    # TODO: apply your move using classical_move

    quantum_move(qc)
    qc.measure_all()
    if run_once(qc) == "0":
        print("Heads! The quantum computer wins.")
    else:
        print("Tails! You beat the quantum computer!")

play_vs_quantum()

## Make it yours

Pick an idea below or invent your own:

- **Best of 5** with a scoreboard and dramatic messages, against the quantum computer.
- **Full quantum duel.** Both players choose from nothing, X, Y, Z, or H on every turn. Build a table of every matchup. Does either side have a winning strategy?
- **Longer games.** What happens with 5 turns instead of 3? Can the Quantum player still always win?
- **Two coins.** Start with an entangled pair (from the warm-up). Design a game where entanglement changes who wins.
- **Teach it.** Turn the Bloch sphere pictures into a step-by-step explanation a non-STEM friend would understand. Great for the Best Quantum Explainer award.

## Explain it

Answer in your own words and be ready to say them out loud.

1. Why doesn't the Classical player's flip change anything after the first H?
2. Why does your counter-move from Mission 4 work?
3. Is the Quantum player cheating? Why or why not?

Our answers: